# SIFLOW · run_8 · DiffusionGemma train + eval + FINAL paper artifacts (SIFLOW-G)

Head-only SIFLOW-G on the DiffusionGemma backbone (6k steps, ~7h), then regenerates ALL tables + figures from every results JSON you import. The downloaded zip holds the final `tables_auto.tex` + `figures/` for the paper.

**Runtime:** designed to finish well under one Colab session. Training stops and checkpoints automatically at an 11h guard — if that happens, just re-run this notebook (re-import its checkpoint) and it resumes.

**How to use:** run every cell top-to-bottom. Cells 1–2 set up the environment and the artifact location; the last cell downloads this part's output zip for the next notebook.

In [ ]:
# === 1. Clone + install (run once per session, ~2 min) ===
REPO_URL = "https://github.com/kagtgi/siflow.git"
import os
if not os.path.isdir("siflow"):
    !git clone $REPO_URL siflow
%cd siflow
!git pull -q
!pip -q install -e .
!pip -q install -r requirements-colab.txt
print("setup done")

In [ ]:
# === 2. Where do artifacts live? ===
# Default: a local folder + zip download/upload between parts (no Drive needed).
# Set USE_DRIVE = True to persist on Google Drive instead (then the import and
# download steps below become no-ops).
USE_DRIVE = False

import os
if USE_DRIVE:
    from siflow.utils import drive
    drive.mount()
    os.environ["SIFLOW_BASE"] = "/content/drive/MyDrive/siflow"
    BASE = drive.base_dir()
else:
    BASE = "/content/artifacts"
    os.makedirs(BASE, exist_ok=True)
print("artifacts ->", BASE)

In [ ]:
# === Hugging Face login ===
# Required for the gated DiffusionGemma weights; recommended for Dream too.
# Get a token at https://huggingface.co/settings/tokens (read scope).
from huggingface_hub import login
login()

### Import the previous part(s)

This part needs the output zip(s) you downloaded earlier. Run the cell below — a file picker opens; select **all** of these at once:

- `run_7_gemma_data.zip` — Gemma-tokenized data (required)
- `run_3_results.zip` — MDLM main-table rows
- `run_4_ablations.zip` — ablation rows
- `run_6_dream.zip` — SIFLOW-D rows

*(If a long run here stopped early at the 11h guard, also re-upload **this** part's own checkpoint zip to resume.)* Using Drive instead? Set `USE_DRIVE=True` above and skip this.

In [ ]:
# === Import previous outputs (pick the .zip files listed above) ===
if not USE_DRIVE:
    from siflow.utils.io import import_zips
    import_zips(BASE)
else:
    print('USE_DRIVE: reading prior outputs from', BASE)

In [ ]:
!python scripts/train.py --config siflow/config/gemma.yaml --set \
    data.tokens_path={BASE}/data/gemma_256.npy \
    out_dir={BASE}/ckpt/gemma run_id=siflow_gemma train.total_steps=6000

In [ ]:
!python scripts/evaluate.py --config siflow/config/gemma.yaml --system siflow \
    --ckpt-dir {BASE}/ckpt/gemma --ref-tokens {BASE}/data/gemma_val.npy \
    --n-samples 400 --k-list 1 4 --out {BASE}/results/gemma.json

In [ ]:
!python scripts/make_tables.py  --results {BASE}/results --out {BASE}/tables_auto.tex
!python scripts/make_figures.py --results {BASE}/results --out-dir {BASE}/figures
print(open(f"{BASE}/tables_auto.tex").read())

In [ ]:
# === Save + auto-download this part's output ===
from siflow.utils.io import export_and_download
if not USE_DRIVE:
    export_and_download(BASE, 'run_8_final_paper_artifacts.zip', include=['results', 'figures', 'tables_auto.tex'])
else:
    print('USE_DRIVE: outputs already persisted under', BASE)

**Done.** Unzip `run_8_final_paper_artifacts.zip` into `paper/` (drop `tables_auto.tex` and `figures/*.pdf` in) and recompile `paper/siflow_aaai.tex` — Tables 2–4 and the figures populate.